# Historical Data Export → Pickles

Pulls all business data from DB1 (old production) and DB2 (current production) into pickle files.

**Output:** `pickle/db2/*.pkl` and `pickle/db1/*.pkl`

**Use cases:** Sales reporting, manifest template mapping, category taxonomy, pricing intelligence.

See `.ai/initiatives/historical_data_export.md` for the full plan.

In [1]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, inspect, text

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


def _resolve_notebook_root() -> Path:
    """Find workspace/notebooks (parent of _shared/) from cwd or NOTEBOOK_DIR."""
    cwd = Path.cwd().resolve()
    if os.environ.get("NOTEBOOK_DIR"):
        p = Path(os.environ["NOTEBOOK_DIR"]).resolve()
        if (p / "_shared" / "config.example.py").is_file():
            return p
        if p.name == "_shared" and (p / "config.example.py").is_file():
            return p.parent
    for base in (cwd, *cwd.parents):
        for candidate in (base, base / "workspace" / "notebooks"):
            if (candidate / "_shared" / "config.example.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not find workspace/notebooks/_shared/config.example.py; "
        "set NOTEBOOK_DIR to workspace/notebooks or run from repo root."
    )


NOTEBOOK_ROOT = _resolve_notebook_root()
SHARED_DIR = NOTEBOOK_ROOT / "_shared"
if str(SHARED_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_DIR))

_cfg = SHARED_DIR / "config_local.py"
if not _cfg.is_file():
    raise FileNotFoundError(
        f"Missing {_cfg} — copy _shared/config.example.py to _shared/config_local.py (see _shared/README.md)."
    )

from config_local import CONNECTIONS  # noqa: E402

# Output directories
HERE = Path.cwd().resolve() if (Path.cwd().resolve() / "export_all.ipynb").exists() else NOTEBOOK_ROOT / "historical-data"
PICKLE_DB2 = HERE / "pickle" / "db2"
PICKLE_DB1 = HERE / "pickle" / "db1"
PICKLE_DB2.mkdir(parents=True, exist_ok=True)
PICKLE_DB1.mkdir(parents=True, exist_ok=True)


def make_engine(cfg: dict):
    user = quote_plus(str(cfg["user"]))
    pw = quote_plus(str(cfg["password"]))
    host = cfg["host"]
    port = int(cfg["port"])
    db = cfg["database"]
    return create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}", future=True)


def open_connection(key: str):
    cfg = CONNECTIONS[key]
    eng = make_engine(cfg)
    label = cfg.get("label", key)
    return eng, label


def fetch_and_save(engine, sql_str, pickle_path, label=""):
    """Run SQL, save pickle, print summary."""
    with engine.connect() as conn:
        df = pd.read_sql_query(text(sql_str), conn)
    df.to_pickle(pickle_path)
    mb = pickle_path.stat().st_size / (1024 * 1024)
    print(f"  {label or pickle_path.name:40s}  {len(df):>8,} rows  {len(df.columns):>3} cols  {mb:>6.1f} MB")
    return df


print("Pickle dirs:")
print(f"  DB2: {PICKLE_DB2}")
print(f"  DB1: {PICKLE_DB1}")

Pickle dirs:
  DB2: E:\ecothrift-dashboard\workspace\notebooks\historical-data\pickle\db2
  DB1: E:\ecothrift-dashboard\workspace\notebooks\historical-data\pickle\db1


---
# DB2 — Production (Aug 2025 → present)

In [2]:
eng2, lab2 = open_connection("production")
print(f"Connected: {lab2}")

Connected: DB2 — Production (local restore / Heroku name may differ)


## DB2 — Sold Items (pre-joined for analysis)
Items with `sold_at IS NOT NULL` joined with product (title/brand/model), manifest_row (category/subcategory), and latest condition from item_history.

In [3]:
DB2_SOLD_ITEMS = """
SELECT
    i.id,
    i.sku,
    p.title,
    COALESCE(p.brand, '')                           AS brand,
    COALESCE(p.model, '')                           AS model,
    COALESCE(mr.category, '')                       AS category,
    COALESCE(mr.subcategory, '')                    AS subcategory,
    i.starting_price,
    i.retail_amt,
    COALESCE(i.sold_for, i.starting_price)          AS sold_for,
    i.sold_at,
    i.on_shelf_at,
    i.processing_completed_at,
    i.created_at,
    COALESCE(i.pricing_type, '')                    AS pricing_type,
    i.disc_a,
    i.disc_b,
    i.inventory_purchase_order_id                   AS purchase_order_id,
    i.manifest_row_id,
    i.product_id,
    COALESCE(
        (SELECT ih.condition
         FROM inventory_item_history ih
         WHERE ih.item_id = i.id
         ORDER BY ih.updated_on DESC
         LIMIT 1),
        ''
    )                                               AS condition
FROM inventory_item i
JOIN inventory_product p ON p.id = i.product_id
LEFT JOIN inventory_manifest_rows mr ON mr.id = i.manifest_row_id
WHERE i.sold_at IS NOT NULL
ORDER BY i.sold_at DESC
"""

df_db2_sold = fetch_and_save(eng2, DB2_SOLD_ITEMS, PICKLE_DB2 / "sold_items.pkl", "db2/sold_items")
display(df_db2_sold.head(3))

  db2/sold_items                              34,762 rows   21 cols    12.8 MB


,id,sku,title,brand,model,category,subcategory,starting_price,retail_amt,sold_for,sold_at,on_shelf_at,processing_completed_at,created_at,pricing_type,disc_a,disc_b,purchase_order_id,manifest_row_id,product_id,condition
0,21214,ITMNCBRBHP,Bink 27oz Day Water Bottle Straw Cap Emerald,Bink,,KITCHEN_AND_DINING,HYDRATION,17.1,34.2,14.73,2026-03-04 12:16:58.698128-06:00,2026-01-17,2025-10-02 12:34:14.627540-05:00,2025-09-29 11:27:59.738661-05:00,discounting,0.15,0.015,58.0,11035.0,11516,good
1,60318,ITMQVN6KBP,Room Essentials 16oz Stoneware Low Key Thriving Mug Microwave Dishwasher Safe,Room Essentials,,KITCHEN_AND_DINING,INLINE MUGS,4.0,8.0,3.45,2026-03-04 12:16:58.698128-06:00,2026-01-17,2025-11-29 13:35:07.955755-06:00,2025-11-24 10:02:59.299462-06:00,discounting,0.15,0.015,159.0,38609.0,40124,good
2,59901,ITMF4NUG8E,"5.8qt Plastic Food Storage Canister Large Pantry Container Clear 13.5"" Height",Brightroom,,KITCHEN_AND_DINING,CANISTERS,8.5,17.0,7.32,2026-03-04 12:16:58.698128-06:00,2026-01-17,2025-11-28 11:38:43.250858-06:00,2025-11-24 10:02:57.494161-06:00,discounting,0.15,0.015,159.0,38461.0,39976,good


## DB2 — All Items

In [4]:
DB2_ALL_ITEMS = """
SELECT
    i.id,
    i.sku,
    p.title,
    COALESCE(p.brand, '')                           AS brand,
    COALESCE(p.model, '')                           AS model,
    COALESCE(mr.category, '')                       AS category,
    COALESCE(mr.subcategory, '')                    AS subcategory,
    i.starting_price,
    i.retail_amt,
    i.sold_for,
    i.sold_at,
    i.on_shelf_at,
    i.processing_completed_at,
    i.created_at,
    COALESCE(i.pricing_type, '')                    AS pricing_type,
    i.disc_a,
    i.disc_b,
    i.bulk_id,
    i.bulk_quantity,
    i.serial_number,
    i.inventory_purchase_order_id                   AS purchase_order_id,
    i.manifest_row_id,
    i.product_id,
    i.product_class_id,
    CASE
        WHEN i.sold_at IS NOT NULL      THEN 'sold'
        WHEN i.on_shelf_at IS NOT NULL  THEN 'on_shelf'
        ELSE 'processing'
    END                                             AS derived_status,
    COALESCE(
        (SELECT ih.condition
         FROM inventory_item_history ih
         WHERE ih.item_id = i.id
         ORDER BY ih.updated_on DESC
         LIMIT 1),
        ''
    )                                               AS condition
FROM inventory_item i
LEFT JOIN inventory_product p ON p.id = i.product_id
LEFT JOIN inventory_manifest_rows mr ON mr.id = i.manifest_row_id
ORDER BY i.id
"""

df_db2_items = fetch_and_save(eng2, DB2_ALL_ITEMS, PICKLE_DB2 / "items.pkl", "db2/items")
display(df_db2_items.head(3))

  db2/items                                   59,833 rows   26 cols    22.6 MB


,id,sku,title,brand,model,category,subcategory,starting_price,retail_amt,sold_for,sold_at,on_shelf_at,processing_completed_at,created_at,pricing_type,disc_a,disc_b,bulk_id,bulk_quantity,serial_number,purchase_order_id,manifest_row_id,product_id,product_class_id,derived_status,condition
0,697,ITMNDMA68E,Liquid Eyebrow Pencil,QIC,,MIXED_LOTS,,40.00,80.00,NaN,None,2026-01-17,2025-08-25 16:24:49.145076-05:00,2025-08-18 19:58:23.568629-05:00,discounting,0.15,0.015,NaN,NaN,NaN,2.0,320.0,320,None,on_shelf,good
1,698,ITMQXDZHGK,Purina Fancy Feast Tender Chicken Feast Kitten,Purina,,MIXED_LOTS,,0.44,0.88,0.45,2025-09-05 19:13:39.305545-05:00,2026-01-17,2025-08-26 09:26:07.132420-05:00,2025-08-18 19:58:23.571336-05:00,discounting,0.15,0.015,NaN,NaN,050000172351,2.0,320.0,2252,None,sold,good
2,699,ITMP6DVAM6,Liquid Eyebrow Pencil,QIC,,MIXED_LOTS,,4.29,8.58,0.92,2025-09-25 12:29:04.232822-05:00,2026-01-17,2025-08-25 16:12:50.794012-05:00,2025-08-18 19:58:23.573693-05:00,discounting,0.15,0.015,NaN,NaN,070686573969,2.0,320.0,320,None,sold,good


## DB2 — Products

In [5]:
df_db2_products = fetch_and_save(eng2, "SELECT * FROM inventory_product ORDER BY id", PICKLE_DB2 / "products.pkl", "db2/products")
display(df_db2_products.head(3))

  db2/products                                41,509 rows   11 cols     8.4 MB


,id,title,brand,model,last_matched_at,match_count,ai_suggested_title,ai_confidence,created_at,updated_at,product_class_id
0,320,Liquid Eyebrow Pencil,QIC,,None,0,,None,2025-08-18 19:58:23.564720-05:00,2025-08-25 16:49:11.516392-05:00,None
1,321,Sumifun Nail Fungus Cream,Sumifun,,None,0,,None,2025-08-18 19:58:23.628303-05:00,2025-08-26 09:38:48.247068-05:00,None
2,322,Seville Classics UltraHD Heavy-Duty Lockable Wall Storage Cabinet with Stain...,Seville Classics,UHD20229B,None,0,,None,2025-08-18 19:58:23.682625-05:00,2025-08-18 19:58:23.682631-05:00,None


## DB2 — Item History (conditions over time)

In [6]:
df_db2_hist = fetch_and_save(eng2, "SELECT * FROM inventory_item_history ORDER BY id", PICKLE_DB2 / "item_history.pkl", "db2/item_history")
display(df_db2_hist.head(3))

  db2/item_history                            54,250 rows   10 cols     6.0 MB


,id,condition,location,status,notes,updated_on,item_id,updated_by_id,destination_id,dispatch_to_id
0,14,very_good,APLE-FLOOR,on_shelf,,2025-08-19 17:37:03.248117-05:00,1411,1,8.0,8.0
1,15,very_good,APLE-FLOOR,on_shelf,,2025-08-19 17:37:42.194202-05:00,1412,1,8.0,8.0
2,16,very_good,APLE-FLOOR,on_shelf,,2025-08-19 17:37:55.486591-05:00,1413,3,8.0,8.0


## DB2 — Purchase Orders

In [7]:
df_db2_po = fetch_and_save(eng2, "SELECT * FROM inventory_purchase_order ORDER BY id", PICKLE_DB2 / "purchase_orders.pkl", "db2/purchase_orders")
display(df_db2_po.head(3))

  db2/purchase_orders                            103 rows   27 cols     0.1 MB


,id,order_number,status,purchase_date,expected_delivery,received_date,retail_value,purchase_price,other_fees,shipping_cost,total_cost,quantity,condition,description,notes,manifest_file_url,manifest_uploaded_at,rows_generated_at,preprocessing_completed_at,processing_started_at,processing_completed_at,created_at,updated_at,created_by_id,csv_template_id,raw_manifest_id,vendor_id
0,2,WAL135876,items_generated,2025-08-14,2025-08-20 09:15:00-05:00,2025-08-20,12386.0,2526.0,101.04,1044.71,3671.75,696,fair,"6 Pallet Spaces of Sporting Goods & More, 696 Units, Ext. Retail $12,386, Sp...",,None,2025-08-18 14:57:30.842678-05:00,None,2025-08-18 19:58:26.340469-05:00,None,None,2025-08-18 14:57:20.432329-05:00,2025-08-22 09:01:16.500013-05:00,1.0,None,NaN,1
1,3,WAL135287,items_generated,2025-08-03,2025-08-10 19:00:00-05:00,2025-08-12,3015.1,15076.0,603.00,2294.97,17973.97,16241,like_new,"Truckload (24 Pallet Spaces) of Mixed Merchandise, 16,241 Units, Like New Co...",,None,2025-08-22 13:32:04.516684-05:00,None,2025-08-25 09:28:00.718308-05:00,None,None,2025-08-22 13:30:58.985250-05:00,2025-09-03 09:46:07.148636-05:00,3.0,None,3.0,1
2,4,TRGET-OKH-2VRC,items_generated,2025-08-05,2025-08-13 09:00:00-05:00,2025-08-14,115392.0,25600.0,1024.00,1950.18,28574.18,4844,good,"Truckload (26 Pallets) of Toys/Games, 4,844 Units, Used - Good Condition, Ex...",Missing 3 Pallets,None,2025-08-25 09:49:19.941681-05:00,None,2025-08-25 10:35:45.898341-05:00,None,None,2025-08-25 09:48:53.376311-05:00,2025-09-03 09:45:35.100907-05:00,1.0,None,4.0,2


## DB2 — Manifest Rows (category/subcategory source)

In [8]:
df_db2_manifest = fetch_and_save(eng2, "SELECT * FROM inventory_manifest_rows ORDER BY id", PICKLE_DB2 / "manifest_rows.pkl", "db2/manifest_rows")
display(df_db2_manifest.head(3))

  db2/manifest_rows                           36,330 rows   18 cols    12.6 MB


,id,row_number,quantity,description,retail_value,brand,model,category,subcategory,upc,sku,vendor_sku,pallet_id,box_id,search_terms,created_at,updated_at,purchase_order_id
0,320,1,25,Generic_GenMerch_$100orless,25.00,,Generic_GenMerch_$100orless,MIXED_LOTS,,7.7777E+11,,7777702-77777,23234313,,,2025-08-18 14:57:39.039992-05:00,2025-08-18 14:57:39.040007-05:00,2
1,321,2,22,Generic_GenMerch_$100orless,25.00,,Generic_GenMerch_$100orless,MIXED_LOTS,,7.7777E+11,,7777702-77777,23112849,,,2025-08-18 14:57:39.041468-05:00,2025-08-18 14:57:39.041479-05:00,2
2,322,3,3,Seville Classics UltraHD Heavy-Duty Lockable Wall Storage Cabinet with Stain...,119.99,,UHD20229B,STORAGE,,1764199229,,370803571-73668,23326872,,HOME IMPROVEMENT,2025-08-18 14:57:39.042421-05:00,2025-08-18 14:57:39.042429-05:00,2


## DB2 — CSV Templates + Field Mappings

In [9]:
df_db2_templates = fetch_and_save(eng2, "SELECT * FROM inventory_csv_template ORDER BY id", PICKLE_DB2 / "csv_templates.pkl", "db2/csv_templates")
display(df_db2_templates)

df_db2_mappings = fetch_and_save(eng2, "SELECT * FROM inventory_csv_field_mapping ORDER BY template_id, \"order\"", PICKLE_DB2 / "csv_field_mappings.pkl", "db2/csv_field_mappings")
display(df_db2_mappings.head(10))

  db2/csv_templates                               10 rows   10 cols     0.0 MB


,id,name,header_signature,is_active,confidence_threshold,created_at,updated_at,created_by_id,updated_by_id,vendor_id
0,1,"Walmart Template - Model, Category, UPC, Item#, PalletID, Dept",UPC|Item #|Model #|Item Description|Category|Department|Qty|Unit Retail|Ext....,True,90,2025-08-18 12:58:00.199623-05:00,2025-08-18 12:58:00.199639-05:00,1,None,1
1,2,Unknown Template - 8/22/2025,location|unit_retail|qty|ext_retail|upc|item|item_description|category|||,True,90,2025-08-22 13:35:37.090509-05:00,2025-08-22 13:35:37.090528-05:00,3,None,1
2,3,"Target - Category, Subcategory, UPC, ITEM #, TCIN, PalletID, Condition",Pallet ID|Item #|Product Class|Category Code|Seller Category|Item Descriptio...,True,90,2025-08-25 09:52:14.742053-05:00,2025-08-25 09:52:14.742069-05:00,1,None,2
3,36,"Costco - Category, Department, Item#, Lot#",Lot #|Item #|Dept. Code|Department|Item Description|Qty|Unit Retail|Ext. Ret...,True,90,2025-08-29 09:05:14.127077-05:00,2025-08-29 09:05:14.127090-05:00,1,None,35
4,37,Unknown Template - 9/3/2025,Lot #|Location|Item #|Item Description|Qty|Unit of Measure|Unit Retail|Ext. ...,True,90,2025-09-03 09:14:25.663555-05:00,2025-09-03 09:14:25.663566-05:00,1,None,36
5,38,"Walmart - Model, Category, Department",UPC|Item #|Model #|Item Description|Category|Department|Qty|Unit Retail|Ext....,True,90,2025-09-16 11:21:21.590956-05:00,2025-09-16 11:21:21.590970-05:00,1,None,1
6,39,Wayfair - 10/1/2025,Pallet ID|Item #|Category|Subcategory|Manufacturer|Item Description|Options|...,True,90,2025-10-01 11:51:25.394730-05:00,2025-10-01 11:51:25.394743-05:00,1,None,37
7,40,Home Depot - 10/1/2025,Order|SB #|Scan LP #|SKU|Item Description|Model #|Qty|Department|Product Typ...,True,90,2025-10-01 13:09:00.244042-05:00,2025-10-01 13:09:00.244056-05:00,1,None,38
8,41,Amazon Template - Everything,Inventory Reference ID|Product Class|GL Description|Category|Subcategory|ASI...,True,90,2025-10-13 10:37:52.771555-05:00,2025-10-13 10:38:19.879048-05:00,1,None,41
9,74,NEw - 12/5/2025,UPC|Walmart Item ID|Unit Retail|Qty|Ext. Retail|Ship Bl Num|Invoice BL|SLP|D...,True,90,2025-12-05 13:46:28.145254-06:00,2025-12-05 13:46:28.145266-06:00,1,None,1


  db2/csv_field_mappings                         120 rows    8 cols     0.0 MB


,id,source_column,target_field,transform_expression,default_value,is_required,order,template_id
0,1,Item Description,description,str(value).strip(),,True,0,1
1,2,Qty,quantity,int(value),1,True,0,1
2,3,Unit Retail,retail_value,float(value),,True,0,1
3,4,,brand,,,False,0,1
4,5,Model #,model,str(value).strip(),,False,0,1
5,6,Category,category,str(value).strip(),,False,0,1
6,7,,subcategory,,,False,0,1
7,8,UPC,upc,str(value).strip(),,False,0,1
8,9,,sku,,,False,0,1
9,10,Item #,vendor_sku,str(value).strip(),,False,0,1


## DB2 — Raw Manifests (S3 refs)

In [10]:
df_db2_raw_man = fetch_and_save(eng2, "SELECT * FROM inventory_raw_manifest ORDER BY id", PICKLE_DB2 / "raw_manifests.pkl", "db2/raw_manifests")
display(df_db2_raw_man.head(3))

  db2/raw_manifests                               61 rows    8 cols     0.0 MB


,id,name,row_count,header_signature,uploaded_at,s3_file_id,uploaded_by_id,vendor_id
0,3,Manifest for PO WAL135287,846,location|unit_retail|qty|ext_retail|upc|item|item_description|category|||,2025-08-22 13:32:04.514401-05:00,5,3,1
1,4,Manifest for PO TRGET-OKH-2VRC,2898,pallet_id|item|product_class|category_code|seller_category|item_description|...,2025-08-25 09:49:19.940105-05:00,8,1,2
2,37,Manifest for PO CST569424,95,lot|item|dept_code|department|item_description|qty|unit_retail|ext_retail|ve...,2025-08-29 09:02:59.668590-05:00,41,1,35


## DB2 — Vendors

In [11]:
df_db2_vendors = fetch_and_save(eng2, "SELECT * FROM inventory_vendor ORDER BY id", PICKLE_DB2 / "vendors.pkl", "db2/vendors")
display(df_db2_vendors)

  db2/vendors                                      9 rows   16 cols     0.0 MB


,id,name,code,vendor_type,contact_name,email,phone,website,address_line1,address_line2,city,state,zip_code,created_at,updated_at,created_by_id
0,1,Walmart,WAL,liquidation,,,,https://liquidations.walmart.com/,,,,,,2025-08-18 17:54:20.384821+00:00,2025-08-18 17:54:20.384836+00:00,1
1,2,Target,TRGET,liquidation,,,,https://bstock.com/buy/seller/target,,,,,,2025-08-25 14:46:58.619332+00:00,2025-08-25 14:46:58.619347+00:00,1
2,35,Costco,CST,liquidation,,,,https://bstock.com/costco,,,,,,2025-08-29 14:01:52.779465+00:00,2025-08-29 14:01:52.779479+00:00,1
3,36,Essendant,ESS,liquidation,,,,https://bstock.com/essendant,,,,,,2025-09-03 14:11:18.743932+00:00,2025-09-03 14:11:18.743944+00:00,1
4,37,Wayfair,WFR,liquidation,,,,https://bstock.com/wayfair,,,,,,2025-10-01 16:48:03.137228+00:00,2025-10-01 16:48:03.137242+00:00,1
5,38,Home Depot,HMD,liquidation,,,,https://bstock.com/homedepot/,,,,,,2025-10-01 18:05:49.136326+00:00,2025-10-01 18:05:49.136338+00:00,1
6,39,Generic,GEN,other,,,,https://www.ecothrift.us/,,,,,,2025-10-10 22:02:42.663476+00:00,2025-10-10 22:02:42.663487+00:00,1
7,40,Ramaekers,RAM,other,Kathy Ramaekers,,,https://www.ecothrift.us/,,,,,,2025-10-13 15:17:19.834233+00:00,2025-10-13 15:17:19.834246+00:00,1
8,41,Amazon,AMZ,liquidation,,,,https://bstock.com/amazon,,,,,,2025-10-13 15:34:35.354357+00:00,2025-10-13 15:34:35.354369+00:00,1


## DB2 — POS: Carts

In [12]:
df_db2_carts = fetch_and_save(eng2, "SELECT * FROM pos_cart ORDER BY id", PICKLE_DB2 / "carts.pkl", "db2/carts")
display(df_db2_carts.head(3))

  db2/carts                                   16,275 rows   13 cols     3.5 MB


,id,status,subtotal,tax_rate,tax_amount,total,created_at,completed_at,updated_at,cashier_id,customer_id,drawer_id,credit_card_fee
0,1,cancelled,536.3,0.07,37.54,573.84,2025-08-23 09:03:28.160487-05:00,None,2025-08-23 09:18:26.540321-05:00,3,None,1,0.0
1,2,active,0.0,0.07,0.00,0.00,2025-08-23 09:15:53.601159-05:00,None,2025-08-23 09:16:03.719384-05:00,3,None,1,0.0
2,3,cancelled,185.8,0.07,13.01,198.81,2025-08-23 09:18:48.983723-05:00,None,2025-08-23 09:19:51.849455-05:00,3,None,1,0.0


## DB2 — POS: Cart Lines

In [13]:
df_db2_lines = fetch_and_save(eng2, "SELECT * FROM pos_cart_line ORDER BY id", PICKLE_DB2 / "cart_lines.pkl", "db2/cart_lines")
display(df_db2_lines.head(3))

  db2/cart_lines                              42,586 rows   11 cols     7.4 MB


,id,quantity,unit_price,line_total,product_title,product_brand,product_model,created_at,cart_id,item_id,discount_rule_id
0,2,1,185.80,185.80,Ladies' Watch - 98L323,Bulova,,2025-08-23 09:17:19.082351-05:00,1,4538.0,NaN
1,3,1,295.61,295.61,Alternator,Unbraded,939184,2025-08-23 09:17:23.886022-05:00,1,4096.0,NaN
2,4,1,25.34,25.34,Easy Touch 5 Dash WindSheild Mount,Unbraded,939184,2025-08-23 09:17:27.423072-05:00,1,5839.0,NaN


## DB2 — POS: Payments

In [14]:
df_db2_payments = fetch_and_save(eng2, "SELECT * FROM pos_payment ORDER BY id", PICKLE_DB2 / "payments.pkl", "db2/payments")
display(df_db2_payments.head(3))

  db2/payments                                15,306 rows    9 cols     1.6 MB


,id,payment_method,amount,amount_tendered,change_given,reference_number,processed_at,cart_id,card_type
0,1,card,31.62,NaN,NaN,,2025-08-23 09:21:22.137061-05:00,4,NaN
1,2,card,31.62,NaN,NaN,,2025-08-23 09:22:29.763106-05:00,5,NaN
2,3,card,236.03,NaN,NaN,,2025-08-23 10:11:11.626755-05:00,9,NaN


## DB2 — POS: Receipts

In [15]:
df_db2_receipts = fetch_and_save(eng2, "SELECT * FROM pos_receipt ORDER BY id", PICKLE_DB2 / "receipts.pkl", "db2/receipts")
display(df_db2_receipts.head(3))

  db2/receipts                                15,306 rows   11 cols     2.1 MB


,id,receipt_number,printed,printed_at,printer_name,emailed,emailed_at,email_address,created_at,cart_id,customer_declined
0,1,R202508230001,True,2025-08-23 09:21:22.358548-05:00,Receipt Printer,False,None,,2025-08-23 09:21:22.148993-05:00,4,False
1,2,R202508230002,True,2025-08-23 09:22:29.962321-05:00,Receipt Printer,False,None,,2025-08-23 09:22:29.773090-05:00,5,False
2,3,R202508230003,True,2025-08-23 10:11:11.864552-05:00,Receipt Printer,False,None,,2025-08-23 10:11:11.650079-05:00,9,False


## DB2 — Processing Details (audit trail)

In [16]:
df_db2_proc = fetch_and_save(eng2, "SELECT * FROM inventory_processing_detail ORDER BY id", PICKLE_DB2 / "processing_details.pkl", "db2/processing_details")
print(f"  Columns: {list(df_db2_proc.columns)}")

  db2/processing_details                      54,611 rows   54 cols    69.5 MB
  Columns: ['id', 'item_id', 'manifest_row_id', 'purchase_order_id', 'init_product_id', 'init_product_title', 'init_product_brand', 'init_product_model', 'init_price_type', 'init_price_starting', 'init_price_disc_a', 'init_price_disc_b', 'init_item_retail', 'final_product_id', 'final_product_title', 'final_product_brand', 'final_product_model', 'final_price_type', 'final_price_starting', 'final_price_disc_a', 'final_price_disc_b', 'final_item_retail', 'manifest_row_number', 'manifest_description', 'manifest_brand', 'manifest_model', 'manifest_expected_qty', 'manifest_retail_value', 'bulk_quantity', 'status', 'product_status', 'pricing_status', 'history_status', 'item_history_condition', 'item_history_location_id', 'item_history_notes', 'item_history_destination_id', 'item_history_dispatch_to_id', 'print_counter', 'processed_at', 'processed_by_id', 'search_terms', 'product_po_items_cnt', 'product_items_cnt', 

## DB2 — Preprocessing Details

In [17]:
df_db2_preproc = fetch_and_save(eng2, "SELECT * FROM inventory_preprocessing_detail ORDER BY id", PICKLE_DB2 / "preprocessing_details.pkl", "db2/preprocessing_details")
print(f"  Columns: {list(df_db2_preproc.columns)}")

  db2/preprocessing_details                   36,330 rows   30 cols    21.1 MB
  Columns: ['id', 'row_number', 'manifest_description', 'manifest_brand', 'manifest_model', 'manifest_retail_value', 'manifest_quantity', 'manifest_match_status', 'suggested_manifest_matches', 'selected_manifest_match_id', 'ai_generated_title', 'ai_generated_brand', 'ai_generated_model', 'ai_generation_status', 'product_match_status', 'suggested_product_matches', 'selected_product_match_id', 'pricing_type', 'starting_price', 'is_bulk', 'bulk_quantity', 'disc_a', 'disc_b', 'pricing_dynamics_status', 'item_rows_generated_count', 'item_generated_status', 'created_at', 'updated_at', 'manifest_row_id', 'purchase_order_id']


## DB2 — Locations + Store Config + Pricing Templates

In [18]:
fetch_and_save(eng2, "SELECT * FROM inventory_location ORDER BY id", PICKLE_DB2 / "locations.pkl", "db2/locations")
fetch_and_save(eng2, "SELECT * FROM inventory_pricing_template ORDER BY id", PICKLE_DB2 / "pricing_templates.pkl", "db2/pricing_templates")
fetch_and_save(eng2, "SELECT * FROM inventory_store_configuration ORDER BY id", PICKLE_DB2 / "store_config.pkl", "db2/store_config")
fetch_and_save(eng2, "SELECT * FROM core_work_location ORDER BY id", PICKLE_DB2 / "work_locations.pkl", "db2/work_locations")
fetch_and_save(eng2, "SELECT * FROM pos_register ORDER BY id", PICKLE_DB2 / "registers.pkl", "db2/registers")
fetch_and_save(eng2, "SELECT * FROM pos_drawer ORDER BY id", PICKLE_DB2 / "drawers.pkl", "db2/drawers")
fetch_and_save(eng2, "SELECT * FROM pos_discount_rule ORDER BY id", PICKLE_DB2 / "discount_rules.pkl", "db2/discount_rules")

  db2/locations                                   11 rows    8 cols     0.0 MB
  db2/pricing_templates                            4 rows    7 cols     0.0 MB
  db2/store_config                                 9 rows    6 cols     0.0 MB
  db2/work_locations                               5 rows   11 cols     0.0 MB
  db2/registers                                    5 rows   18 cols     0.0 MB
  db2/drawers                                    229 rows   10 cols     0.1 MB
  db2/discount_rules                               2 rows   13 cols     0.0 MB


,id,name,button_text,description,discount_type,discount_value,min_purchase,is_active,start_date,end_date,created_at,updated_at,created_by_id
0,1,GRAND OPENING: $15 Off $100 Purchase,$15 off $100,"GRAND OPENING: $15 Off $100 Purchase\nCanfield on October 11th - 12th, 2025",flat_amount,15.0,100.0,True,2025-10-10 05:00:00+00:00,2025-10-13 04:55:00+00:00,2025-10-11 01:24:57.158262+00:00,2025-12-08 10:25:45.339281-06:00,1
1,2,"GRAND OPENING: $200 Off $1,000 Purchase","$200 off $1,000","GRAND OPENING - $200 off $1,000 purchase\nCanfield on October 11th - 12th, 2025",flat_amount,200.0,1000.0,True,2025-10-10 05:00:00+00:00,2025-10-12 16:00:00+00:00,2025-10-11 01:27:17.411230+00:00,2025-10-10 20:27:41.642439-05:00,1


In [19]:
eng2.dispose()
print("DB2 engine disposed. All DB2 pickles saved.")

DB2 engine disposed. All DB2 pickles saved.


---
# DB1 — Old Production (Feb 2023 → ~Aug 2025)

In [20]:
eng1, lab1 = open_connection("old_production")
print(f"Connected: {lab1}")

Connected: DB1 — Old Production (archive)


## DB1 — Sold Items (pre-joined for analysis)
Cart lines for completed non-void carts, joined with item + product + product_attrs (category/subcategory).

In [21]:
DB1_SOLD_ITEMS = """
SELECT
    cl.id                                            AS cart_line_id,
    c.code                                           AS cart_code,
    c.close_time                                     AS sold_at,
    c.open_time,
    c.subtotal_amt                                   AS cart_subtotal,
    c.tax_amt                                        AS cart_tax,
    c.total_amt                                      AS cart_total,
    c.total_discount_amt                              AS cart_discount,
    c.drawer_cde,
    c.void,
    cl.item_cde                                      AS sku,
    cl.line_description,
    cl.quantity,
    cl.unit_price_amt,
    cl.total_price_amt                               AS line_total,
    cl.taxable,
    cl.is_standard_bogo,
    i.id                                             AS item_id,
    i.starting_price_amt,
    i.retail_amt,
    i.is_static,
    i.order_number,
    i.product_cde,
    i.bulk_cde,
    COALESCE(pr.title, '')                           AS product_title,
    COALESCE(pr.brand, '')                           AS brand,
    COALESCE(pr.model, '')                           AS model,
    COALESCE(pa.category, '')                        AS category,
    COALESCE(pa.subcategory, '')                     AS subcategory,
    COALESCE(pa.upc, '')                             AS upc,
    pa.retail_amt                                    AS product_retail_amt,
    pa.taxable                                       AS product_taxable
FROM cart_line cl
JOIN cart c ON c.code = cl.cart_cde
LEFT JOIN item i ON i.code = cl.item_cde
LEFT JOIN product pr ON pr.code = i.product_cde
LEFT JOIN product_attrs pa ON pa.product_cde = i.product_cde
WHERE c.close_time IS NOT NULL
  AND c.void = false
ORDER BY c.close_time DESC
"""

df_db1_sold = fetch_and_save(eng1, DB1_SOLD_ITEMS, PICKLE_DB1 / "sold_items.pkl", "db1/sold_items")
display(df_db1_sold.head(3))

  db1/sold_items                            1,933,940 rows   32 cols   774.4 MB


,cart_line_id,cart_code,sold_at,open_time,cart_subtotal,cart_tax,cart_total,cart_discount,drawer_cde,void,sku,line_description,quantity,unit_price_amt,line_total,taxable,is_standard_bogo,item_id,starting_price_amt,retail_amt,is_static,order_number,product_cde,bulk_cde,product_title,brand,model,category,subcategory,upc,product_retail_amt,product_taxable
0,89304,jkBV78m0g,9999-12-30 18:00:00-06:00,2024-04-14 12:19:15-05:00,0.02,0.00,0.02,0.0,Wzl0dA7Th,False,2q51fdYOw,Iphone 6.7in Black Case,1,0.01,0.01,True,False,1710.0,14.99,19.99,False,CST419759,0kl89dyfg,,Iphone 6.7in Black Case,Pelican,,Electronics & Technology,Phones & Accessories,,19.99,True
1,79420,LfT0k7bn5,9999-12-30 18:00:00-06:00,2024-02-22 05:42:43-06:00,34.00,2.38,36.38,0.0,5Og5yAGJh,False,DwaFXKyLQ,"DUDE Wipes - Flushable Wipes - 3 Pack, 144 Wipes - Mint Chill Extra-Large Ad...",1,4.00,4.00,True,False,NaN,NaN,NaN,None,NaN,NaN,NaN,,,,,,,NaN,None
2,87839,Ckc39Pgux,9999-12-30 18:00:00-06:00,2024-03-22 04:36:27-05:00,34.75,2.43,37.18,0.0,Fzp8kZTVa,False,6XWVdVz30,Chair,2,5.00,10.00,True,False,1694.0,NaN,NaN,True,NaN,HJdOpfXKv,NaN,Unknown Item,,,,,,NaN,True


## DB1 — All Items (with product + product_attrs)

In [22]:
DB1_ALL_ITEMS = """
SELECT
    i.id,
    i.code                                           AS sku,
    i.starting_price_amt,
    i.retail_amt,
    i.is_static,
    i.quantity,
    i.order_number,
    i.line_number,
    i.product_cde,
    i.bulk_cde,
    i.price_lbl,
    COALESCE(pr.title, '')                           AS product_title,
    COALESCE(pr.brand, '')                           AS brand,
    COALESCE(pr.model, '')                           AS model,
    COALESCE(pa.category, '')                        AS category,
    COALESCE(pa.subcategory, '')                     AS subcategory,
    COALESCE(pa.upc, '')                             AS upc,
    pa.retail_amt                                    AS product_retail_amt,
    pa.taxable                                       AS product_taxable
FROM item i
LEFT JOIN product pr ON pr.code = i.product_cde
LEFT JOIN product_attrs pa ON pa.product_cde = i.product_cde
ORDER BY i.id
"""

df_db1_items = fetch_and_save(eng1, DB1_ALL_ITEMS, PICKLE_DB1 / "items.pkl", "db1/items")
display(df_db1_items.head(3))

  db1/items                                 2,890,472 rows   19 cols   621.7 MB


,id,sku,starting_price_amt,retail_amt,is_static,quantity,order_number,line_number,product_cde,bulk_cde,price_lbl,product_title,brand,model,category,subcategory,upc,product_retail_amt,product_taxable
0,1,qthFHRXwu,161.45,169.95,False,1,AMZ11175,1.0,mksNtyUkK,,H9,"Shun Classic Chef's Knife - 8in, Thin, Light",Shun Cutlery,,Home & Decor,Kitchenware,8939865242886,169.95,True
1,2,k2O1NIkea,84.55,89.00,False,1,AMZ11175,2.0,mKX9OKzMV,,G7,Keurig K Supreme,Keurig,,Home & Decor,Appliances,61124738833,89.00,True
2,2,k2O1NIkea,84.55,89.00,False,1,AMZ11175,2.0,mKX9OKzMV,,G7,Keurig K Supreme,Keurig,,Home & Decor,Appliances,61124738833,124.00,True


## DB1 — Products

In [23]:
df_db1_products = fetch_and_save(eng1, "SELECT * FROM product ORDER BY id", PICKLE_DB1 / "products.pkl", "db1/products")
display(df_db1_products.head(3))

  db1/products                               140,621 rows    5 cols    11.3 MB


,id,code,title,brand,model
0,1,mksNtyUkK,"Shun Classic Chef's Knife - 8in, Thin, Light",Shun Cutlery,
1,2,mKX9OKzMV,Keurig K Supreme,Keurig,
2,3,QZCKFL04a,6-Cup Stovetop Espresso Maker - Satin Stainless Steel,Cuisinox,Roma


## DB1 — Product Attrs (category taxonomy)

In [24]:
df_db1_pa = fetch_and_save(eng1, "SELECT * FROM product_attrs ORDER BY id", PICKLE_DB1 / "product_attrs.pkl", "db1/product_attrs")
display(df_db1_pa.head(3))

print("\nCategory distribution (top 20):")
display(df_db1_pa["category"].value_counts().head(20))
print("\nSubcategory distribution (top 20):")
display(df_db1_pa["subcategory"].value_counts().head(20))

  db1/product_attrs                          152,983 rows   10 cols    21.4 MB


,id,product_cde,upc,category,subcategory,retail_amt,attrs,quantity,last_instance,taxable
0,1,mksNtyUkK,8939865242886,Home & Decor,Kitchenware,169.95,NaN,1.0,2024-03-14 08:01:52.825979-05:00,True
1,2,mKX9OKzMV,61124738833,Home & Decor,Appliances,89.00,NaN,1.0,2024-03-14 08:01:52.825979-05:00,True
2,3,QZCKFL04a,6285435218,Home & Decor,Kitchenware,84.38,NaN,1.0,2024-03-14 08:01:52.825979-05:00,True



Category distribution (top 20):


category
Home & Decor                      62805
Toys, Games & Arts                23461
Home Improvement & Tools          11169
Pets & Animal Care                 9466
Beauty, Health & Personal Care     8521
Electronics & Technology           8293
Sports, Fitness & Outdoors         8123
Clothing, Shoes & Accessories      5257
Media & Entertainment              3959
Office & School Supplies           3099
Miscellaneous & Uncategorized      2903
Garden, Patio & Outdoor Living     2311
Automotive & Garage                1696
Kids Home                           603
Baby & Toddler Toys                 166
Bedding & Bath                      123
Seasonal & Holiday                  105
Kids' Bedding & Decor               100
Baby Products                        96
Party & Celebration                  79
Name: count, dtype: int64


Subcategory distribution (top 20):


subcategory
Home Decor                13905
Kitchenware               13339
Bedding & Bath            13177
Storage & Organization    10989
Dolls & Action Figures     9039
Dogs                       6847
Appliances                 4630
Furniture                  4289
Games & Puzzles            3704
Phones & Accessories       3439
Personal Care              3324
Baby & Toddler Toys        3055
Hardware                   2711
Seasonal & Holiday         2664
Outdoor Recreation         2585
Outdoor Play               2555
Arts & Crafts              2429
Team Sports                2342
Bags & Luggage             2311
Computers                  2233
Name: count, dtype: int64

## DB1 — Purchase Orders

In [25]:
df_db1_po = fetch_and_save(eng1, "SELECT * FROM purchase_order ORDER BY id", PICKLE_DB1 / "purchase_orders.pkl", "db1/purchase_orders")
display(df_db1_po.head(3))

  db1/purchase_orders                            210 rows   18 cols     0.1 MB


,id,number,created_on,retail_amt,condition_id,quantity,description,price_amt,fee_amt,shipping_amt,paid_amt,delivery_address_cde,scheduled_delivery,received_on,purchased_on,paid_on,preprocessed_on,processed_on
0,1,AMZ11175,2024-03-14 07:28:55-05:00,6720.0,5,313,"Est. 2 Pallets of Home Goods by Brita, Hamilton Beach & More, 313 Units, Use...",1596.0,0.00,209.99,1805.99,WTx5czCwn,2024-03-21 07:28:55-05:00,2024-03-21 07:28:55-05:00,2024-03-14 07:28:55-05:00,2024-03-14 07:28:55-05:00,2024-03-14 07:28:55-05:00,2024-03-24 07:28:55-05:00
1,2,HMD51060,2024-03-14 07:28:55-05:00,11148.0,5,117,"5 Pallets of Tools, Outdoor Items, Storage & Organization, Automotive Suppli...",2275.0,68.25,1673.38,4016.63,WTx5czCwn,2024-03-02 03:22:00-06:00,2024-03-02 03:22:00-06:00,2024-03-14 07:28:55-05:00,2024-03-14 07:28:55-05:00,2024-03-14 07:28:55-05:00,2024-03-24 07:28:55-05:00
2,3,TGT100676,2024-03-14 07:28:55-05:00,35757.0,4,989,"1 Pallet of Electronics & Accessories, Used - Good Condition, 989 Units, Ext...",3630.0,108.90,264.62,4003.52,WTx5czCwn,2024-03-05 07:08:00-06:00,2024-03-05 07:08:00-06:00,2024-03-14 07:28:55-05:00,2024-03-14 07:28:55-05:00,2024-03-14 07:28:55-05:00,2024-03-24 07:28:55-05:00


## DB1 — Manifests (with category/subcategory)

In [26]:
df_db1_manifest = fetch_and_save(eng1, "SELECT * FROM manifest ORDER BY id", PICKLE_DB1 / "manifests.pkl", "db1/manifests")
display(df_db1_manifest.head(3))

print("\nManifest category distribution (top 20):")
display(df_db1_manifest["category"].value_counts().head(20))

  db1/manifests                              107,748 rows   18 cols    28.9 MB


,id,order_number,line_number,quantity,retail_amt,ext_retail_amt,description,brand,model,category,subcategory,pallet_id,upc,sku,search_string_1,search_string_2,search_string_3,product_cde
0,1,AMZ11175,1,1,169.95,169.95,"Shun Cutlery Classic Chef'S Knife 8”, Thin, Light Kitchen Knife, Ideal For A...",,,Food Service,Commercial Kitchen Supplies & Utensils,,8939865242886,,B0000Y7KNQ,402efc18-58bd-43cc-97b0-ba228d41a840,,mksNtyUkK
1,2,AMZ11175,2,1,89.00,89.00,"Keurig K-Mini Single Serve Coffee Maker, Oasis",,,Hot Beverages,Single Serve Brewers,,61124738833,,B07GV2BHKC,ddcb63bf-179f-48cb-a123-2725cba415ef,,mKX9OKzMV
2,3,AMZ11175,3,1,84.38,84.38,"Cuisinox Roma Satin Stainless Steel Moka Pot Stovetop Espresso Maker, 6-Cup",,,Hot Beverages,Espresso Machines,,6285435218,,B08WYCF2F4,e3ebd7c2-f320-48f9-879b-eb3b7d2c2e4a,,QZCKFL04a



Manifest category distribution (top 20):


category
Hardware And Tools            5901
Gamesdiecastaction Figures    3661
Storageorganization           2872
Tabletop                      2781
Basic Bedding                 2754
Bullseyes Playground          2200
Hearth And Hand               2071
Kids Home                     2021
Seasonal                      1995
Activitiesdollsplush          1963
Lightingwall Decor            1914
Luggage                       1866
Collection Bedding            1825
Bath                          1636
Kitchen                       1618
Window                        1510
Soft Decor                    1508
Kitchenware                   1487
Home                          1471
Schooloffice Supplies         1388
Name: count, dtype: int64

## DB1 — Carts

In [27]:
df_db1_carts = fetch_and_save(eng1, "SELECT * FROM cart ORDER BY id", PICKLE_DB1 / "carts.pkl", "db1/carts")
display(df_db1_carts.head(3))

  db1/carts                                   53,304 rows   17 cols    10.8 MB


,id,code,drawer_cde,open_time,close_time,subtotal_amt,total_discount_amt,total_taxable_amt,sales_tax_percentage,tax_amt,total_amt,total_payment_amt,notes,void,void_reason,replaced_cart_cde,thrift_plus_member_id
0,24669,zu2BTkCej,1BJZsFYGV,2023-11-17 03:21:26-06:00,2023-11-17 03:53:03-06:00,26.99,0.0,26.99,7.0,1.89,28.88,30.00,NaN,False,NaN,None,NaN
1,24670,ZCtWaYYGk,1BJZsFYGV,2023-11-17 03:53:03-06:00,2023-11-17 04:11:07-06:00,14.49,0.0,14.49,7.0,1.02,15.90,15.90,NaN,False,NaN,None,NaN
2,24671,reMkpMg2R,1BJZsFYGV,2023-11-17 04:11:07-06:00,2023-11-17 05:24:51-06:00,60.00,0.0,60.00,7.0,4.20,65.81,65.81,NaN,False,NaN,None,NaN


## DB1 — Cart Lines

In [28]:
df_db1_lines = fetch_and_save(eng1, "SELECT * FROM cart_line ORDER BY id", PICKLE_DB1 / "cart_lines.pkl", "db1/cart_lines")
display(df_db1_lines.head(3))

  db1/cart_lines                             173,475 rows   10 cols    16.5 MB


,id,cart_cde,item_cde,line_num,line_description,quantity,unit_price_amt,total_price_amt,taxable,is_standard_bogo
0,56676,12k8WrkAH,Nx7vRNkpw,1,$5 Cashier Bin,1,5.0,5.0,True,False
1,56677,12s9Jc6EP,1sRvbC7Sm,1,Quantum Stainless Steel Bottom Sink Grid,1,16.0,16.0,True,False
2,56678,12zTMEsZu,gwV1Be32n,1,Unknown Item,1,10.0,10.0,True,False


## DB1 — Payments

In [29]:
df_db1_payments = fetch_and_save(eng1, "SELECT * FROM payment ORDER BY id", PICKLE_DB1 / "payments.pkl", "db1/payments")
display(df_db1_payments.head(3))

  db1/payments                                55,176 rows    7 cols     3.5 MB


,id,cart_cde,type,amount,fee_amount,pmt_string,giftcard_cde
0,25612,7uT92EBnP,Cash,8.00,0.00,NaN,NaN
1,25613,Prvzv61zX,Cash,10.00,0.00,NaN,NaN
2,25614,2GpR52YAk,Credit,82.13,1.88,NaN,NaN


## DB1 — Prices (price schedule + index)

In [30]:
df_db1_prices = fetch_and_save(eng1, "SELECT * FROM price ORDER BY id", PICKLE_DB1 / "prices.pkl", "db1/prices")
display(df_db1_prices.head(5))

df_db1_price_idx = fetch_and_save(eng1, "SELECT * FROM price_index ORDER BY code, index", PICKLE_DB1 / "price_index.pkl", "db1/price_index")
display(df_db1_price_idx.head(5))

  db1/prices                                  10,816 rows    5 cols     0.4 MB


,id,code,week_num,amount,weeks_to_zero
0,1,A4,0,50000.0,104
1,2,B4,0,47000.0,103
2,3,C4,0,44000.0,102
3,4,D4,0,41000.0,101
4,5,E4,0,38000.0,100


  db1/price_index                                234 rows    4 cols     0.0 MB


,index,code,amount,label
0,233,A1,0.0,Free
1,232,A2,0.0,Free
2,231,A3,0.0,Free
3,230,A4,0.0,Free
4,229,A5,0.0,Free


## DB1 — Condition + Status lookup tables

In [31]:
df_db1_list_cond = fetch_and_save(eng1, "SELECT * FROM list_condition ORDER BY id", PICKLE_DB1 / "list_condition.pkl", "db1/list_condition")
display(df_db1_list_cond)

df_db1_list_stat = fetch_and_save(eng1, "SELECT * FROM list_status ORDER BY id", PICKLE_DB1 / "list_status.pkl", "db1/list_status")
display(df_db1_list_stat)

  db1/list_condition                               7 rows    2 cols     0.0 MB


,id,condition_name
0,1,New
1,2,Like New
2,3,Very Good
3,4,Good
4,5,Fair
5,6,Repairable
6,7,Parts Only


  db1/list_status                                 27 rows    2 cols     0.0 MB


,id,status_name
0,1,PreProcess - Purchased
1,2,PreProcess - Unfulfilled
2,3,PreProcess - Undelivered
3,4,PreProcess - Order Received
4,5,CPU - Processed
5,6,Restoration - Tested
6,7,Restoration - Assembled
7,8,Restoration - Repaired / Serviced
8,9,Restoration - Salvaged / Disposed
9,10,CPU - Prepped


## DB1 — Item Conditions (history)

In [32]:
df_db1_item_cond = fetch_and_save(eng1, "SELECT * FROM item_condition ORDER BY id", PICKLE_DB1 / "item_conditions.pkl", "db1/item_conditions")
print(f"  {len(df_db1_item_cond):,} condition records")

  db1/item_conditions                        130,112 rows    5 cols    11.4 MB
  130,112 condition records


## DB1 — Item Statuses (history)

In [33]:
df_db1_item_stat = fetch_and_save(eng1, "SELECT * FROM item_status ORDER BY id", PICKLE_DB1 / "item_statuses.pkl", "db1/item_statuses")
print(f"  {len(df_db1_item_stat):,} status records")

  db1/item_statuses                          217,305 rows    5 cols    19.0 MB
  217,305 status records


## DB1 — Drawers

In [34]:
df_db1_drawers = fetch_and_save(eng1, "SELECT * FROM drawer ORDER BY id", PICKLE_DB1 / "drawers.pkl", "db1/drawers")
display(df_db1_drawers.head(3))

  db1/drawers                                    918 rows    9 cols     0.1 MB


,id,code,cashier_cde,open_manager_cde,close_manager_cde,open_time,close_time,open_cash_amt,close_cash_amt
0,19,km7SjB6rA,GGYCk4C6H,tjneVzTE8,tjneVzTE8,2023-02-20 22:28:16-06:00,2023-02-21 06:28:16-06:00,86.0,95.63
1,20,fPfwy8GRF,vnnMC53Mz,tjneVzTE8,tjneVzTE8,2023-02-21 03:01:59-06:00,2023-02-21 11:01:59-06:00,28.0,201.34
2,24,GmxkaLG41,GGYCk4C6H,,,2023-12-11 03:55:13-06:00,1899-12-29 18:00:00-06:00,200.0,0.00


## DB1 — Discounts

In [35]:
fetch_and_save(eng1, "SELECT * FROM discount ORDER BY id", PICKLE_DB1 / "discounts.pkl", "db1/discounts")
fetch_and_save(eng1, "SELECT * FROM cart_discount ORDER BY id", PICKLE_DB1 / "cart_discounts.pkl", "db1/cart_discounts")

  db1/discounts                                    3 rows   13 cols     0.0 MB
  db1/cart_discounts                           1,505 rows    6 cols     0.1 MB


,id,cart_cde,discount_cde,cart_label,cart_value,taxable
0,3,700fskKoY,99OFFSALE,99% OFF SALE - -0.16% OFF,0.00,True
1,4,bobl3OZHj,99OFFSALE,99% OFF SALE - -0.15% OFF,0.00,True
2,5,nXyWIBpH7,99OFFSALE,99% OFF SALE - 0.49% OFF,-0.13,True
3,6,IP2GmQA5O,99OFFSALE,99% OFF SALE - 0.5% OFF,-0.15,True
4,7,zVDI0YxQa,99OFFSALE,99% OFF SALE - 0.91% OFF,-0.05,True
...,...,...,...,...,...,...
1500,1537,UqSPvDnSW,BGO241015,BOGO - Direct Mailer 10.15.24,-2.60,True
1501,1538,uB2qA9UUu,BGO241015,BOGO - Direct Mailer 10.15.24,-4.56,True
1502,1539,sbmmKixvE,BGO241015,BOGO - Direct Mailer 10.15.24,-16.42,True
1503,1540,JiDdpfuFi,BGO241015,BOGO - Direct Mailer 10.15.24,-10.00,True


In [36]:
eng1.dispose()
print("DB1 engine disposed. All DB1 pickles saved.")

DB1 engine disposed. All DB1 pickles saved.


---
# Summary

In [37]:
import json
from datetime import datetime, timezone

manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(), "pickles": {}}

total_mb = 0
print("=" * 75)
print("  PICKLE MANIFEST")
print("=" * 75)
for db_dir, db_label in [(PICKLE_DB2, "db2"), (PICKLE_DB1, "db1")]:
    print(f"\n--- {db_label.upper()} ---")
    for pkl in sorted(db_dir.glob("*.pkl")):
        df = pd.read_pickle(pkl)
        mb = pkl.stat().st_size / (1024 * 1024)
        total_mb += mb
        key = f"{db_label}/{pkl.stem}"
        manifest["pickles"][key] = {"rows": len(df), "cols": len(df.columns), "mb": round(mb, 2)}
        print(f"  {key:45s}  {len(df):>8,} rows  {len(df.columns):>3} cols  {mb:>6.1f} MB")

print(f"\n{'='*75}")
print(f"  Total: {len(manifest['pickles'])} files, {total_mb:.1f} MB")
print(f"{'='*75}")

manifest_path = HERE / "pickle" / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"\nManifest written: {manifest_path}")

  PICKLE MANIFEST

--- DB2 ---
  db2/cart_lines                                   42,586 rows   11 cols     7.4 MB
  db2/carts                                        16,275 rows   13 cols     3.5 MB
  db2/csv_field_mappings                              120 rows    8 cols     0.0 MB
  db2/csv_templates                                    10 rows   10 cols     0.0 MB
  db2/discount_rules                                    2 rows   13 cols     0.0 MB
  db2/drawers                                         229 rows   10 cols     0.1 MB
  db2/item_history                                 54,250 rows   10 cols     6.0 MB
  db2/items                                        59,833 rows   26 cols    22.6 MB
  db2/locations                                        11 rows    8 cols     0.0 MB
  db2/manifest_rows                                36,330 rows   18 cols    12.6 MB
  db2/payments                                     15,306 rows    9 cols     1.6 MB
  db2/preprocessing_details                  

In [38]:
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

_cwd = Path.cwd().resolve()
if (_cwd / "export_all.ipynb").is_file():
    HERE = _cwd
else:
    HERE = _cwd
    for p in [_cwd, *_cwd.parents]:
        cand = p / "historical-data"
        if (cand / "export_all.ipynb").is_file():
            HERE = cand
            break
PICKLE_ROOT = HERE / "pickle"
DB_DIRS = [("db2", PICKLE_ROOT / "db2"), ("db1", PICKLE_ROOT / "db1")]

manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(), "pickles": {}}
total_mb = 0

print(f"{'File':<45s}  {'Rows':>8s}  {'Cols':>4s}  {'MB':>7s}  Column Names")
print("-" * 130)

for db_label, db_dir in DB_DIRS:
    if not db_dir.exists():
        continue
    for pkl in sorted(db_dir.glob("*.pkl")):
        df = pd.read_pickle(pkl)
        mb = pkl.stat().st_size / (1024 * 1024)
        total_mb += mb
        key = f"{db_label}/{pkl.stem}"
        columns = list(df.columns)
        manifest["pickles"][key] = {
            "rows": len(df),
            "cols": len(columns),
            "mb": round(mb, 2),
            "columns": columns,
        }
        col_preview = ", ".join(columns[:8])
        if len(columns) > 8:
            col_preview += f", ... (+{len(columns) - 8} more)"
        print(f"  {key:<43s}  {len(df):>8,}  {len(columns):>4}  {mb:>6.1f}  {col_preview}")

print(f"\n{'='*130}")
print(f"  Total: {len(manifest['pickles'])} files, {total_mb:.1f} MB")
print(f"{'='*130}")

manifest_path = PICKLE_ROOT / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"\nManifest written: {manifest_path}")

File                                               Rows  Cols       MB  Column Names
----------------------------------------------------------------------------------------------------------------------------------
  db2/cart_lines                                 42,586    11     7.4  id, quantity, unit_price, line_total, product_title, product_brand, product_model, created_at, ... (+3 more)
  db2/carts                                      16,275    13     3.5  id, status, subtotal, tax_rate, tax_amount, total, created_at, completed_at, ... (+5 more)
  db2/csv_field_mappings                            120     8     0.0  id, source_column, target_field, transform_expression, default_value, is_required, order, template_id
  db2/csv_templates                                  10    10     0.0  id, name, header_signature, is_active, confidence_threshold, created_at, updated_at, created_by_id, ... (+2 more)
  db2/discount_rules                                  2    13     0.0  id, name, but